# 아이차트 순위 크롤링 → Claude 카드뉴스 → 엑셀 저장

`t01_ichart_demo.py`를 수업용으로 단계별로 나눈 노트북입니다. 위에서부터 셀을 하나씩 실행하세요.

| 단계 | 하는 일 |
|---|---|
| ① 수집 | 주소 → HTML 글자 → 순위 표 (requests, BeautifulSoup) |
| ② 변환 | 순위 표 → Claude → 카드뉴스 문구 (anthropic) |
| ③ 저장 | 순위 + 카드뉴스 → 엑셀 파일 (pandas) |

> 참고: 이 페이지의 기본 탭은 **Live(실시간)** 입니다. Weekly 탭은 자바스크립트로 불러와서 `requests`로는 받을 수 없습니다.

## 0. 준비: 라이브러리와 설정

처음 한 번만 터미널에서 설치합니다: `pip install -r requirements.txt`

Claude를 쓰려면 이 폴더에 `.env` 파일을 만들고 `ANTHROPIC_API_KEY=...` 한 줄을 넣습니다 (`.env.example` 참고).

In [1]:
from pathlib import Path

from dotenv import load_dotenv

BASE_DIR = Path.cwd()          # 노트북이 있는 폴더
load_dotenv(BASE_DIR / ".env")  # ANTHROPIC_API_KEY 읽기

ICHART_URL = "https://www.ichart.kr/rank"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
TOP_N = 10

## ① 수집 1: 주소 → HTML 글자

`User-Agent`를 넣어 브라우저처럼 요청합니다. 상태 코드가 200이면 성공입니다.

In [2]:
import requests

res = requests.get(ICHART_URL, headers=HEADERS, timeout=10)
res.raise_for_status()
res.encoding = "utf-8"
html = res.text

print("상태 코드:", res.status_code)
print("글자 수:", len(html))

상태 코드: 200
글자 수: 379049


받은 HTML을 파일로 저장해 두면, 나중에 사이트가 바뀌거나 인터넷이 안 될 때도 같은 실습을 할 수 있습니다.

In [3]:
data_dir = BASE_DIR / "data"
data_dir.mkdir(exist_ok=True)
(data_dir / "ichart_rank.html").write_text(html, encoding="utf-8")

# 인터넷이 안 될 때는 위 셀 대신 아래 줄의 주석을 풀어 저장해 둔 파일을 읽기
# html = (data_dir / "ichart_rank.html").read_text(encoding="utf-8")

379049

## ① 수집 2: F12로 구조 확인 → 선택자 정하기

브라우저에서 F12 → 요소 선택으로 순위 한 줄을 눌러 보면 이런 구조입니다.

```html
<div aria-labelledby="ranking-title">          ← 순위 영역
  <a href="/song/52712">                        ← 곡 하나
    <span>1</span>                               ← 순위
    <img alt="noChange">                         ← 순위 변동 아이콘 (up / down / noChange)
    <img alt="NEON 앨범 커버 — A2O, A2O LITE">   ← 앨범 커버: 곡명과 가수가 alt에!
    <h3>NEON</h3>  <strong>199</strong>          ← 곡명, 점수
    <p>A2O, A2O LITE</p>                          ← 가수
  </a>
```

**주의**: 링크 안 첫 번째 `<img>`는 앨범 커버가 아니라 **변동 아이콘**입니다.
`a.select_one("img")`로 가져오면 엉뚱한 이미지를 잡아 0곡이 나옵니다. 그래서 alt로 골라 잡습니다.

In [4]:
from bs4 import BeautifulSoup

RANK_AREA = '[aria-labelledby="ranking-title"]'  # "iChart Ranking" 영역만
SONG_LINK = 'a[href*="/song/"]'                   # 곡 하나 = /song/번호 링크
ALT_SEP = " 앨범 커버 — "                          # 커버 alt: "곡명 앨범 커버 — 가수"
COVER_IMG = 'img[alt*="앨범 커버"]'                # 앨범 커버 이미지
CHANGE_IMG = 'img:not([alt*="앨범 커버"])'         # 변동 아이콘
SCORE_SEL = "b, strong"                           # 굵은 숫자(점수)

soup = BeautifulSoup(html, "html.parser")
area = soup.select_one(RANK_AREA)
links = area.select(SONG_LINK)
print("순위 영역 안 곡 링크 수:", len(links))

순위 영역 안 곡 링크 수: 10


선택자가 맞는지 첫 번째 곡 하나만 꺼내서 확인해 봅니다.

In [5]:
a = links[0]
print("링크  :", a.get("href"))
print("첫 img:", a.select_one("img").get("alt"), " ← 변동 아이콘")
print("커버  :", a.select_one(COVER_IMG).get("alt"))
print("점수  :", a.select_one(SCORE_SEL).get_text(strip=True))

링크  : /song/52712
첫 img: noChange  ← 변동 아이콘
커버  : NEON 앨범 커버 — A2O, A2O LITE
점수  : 211


## ① 수집 3: 곡마다 반복해서 표 만들기

- 커버 alt를 `ALT_SEP`로 나누면 곡명과 가수가 됩니다.
- 변동 아이콘 alt(`up`/`down`/`noChange`)와 옆 숫자를 합쳐 `▲1`, `▼1`, `-`로 바꿉니다.
- 같은 곡 링크가 두 번 나올 수 있으므로 `seen`으로 중복을 거릅니다.

In [6]:
from urllib.parse import urljoin

import pandas as pd

CHANGE_MARK = {"up": "▲", "down": "▼", "noChange": "-"}


# tag 안에서 selector에 맞는 첫 요소의 글자, 없으면 빈칸
def text_of(tag, selector):
    found = tag.select_one(selector)
    return found.get_text(strip=True) if found else ""


# 순위 변동: 아이콘 alt + 옆 숫자 → "▲1", "▼2", "-"
def change_of(a):
    img = a.select_one(CHANGE_IMG)
    if img is None:
        return ""
    mark = CHANGE_MARK.get(img.get("alt", ""), img.get("alt", ""))
    return mark + img.parent.get_text(strip=True)


# HTML 글자 → 순위 DataFrame
def parse_ranking(html):
    soup = BeautifulSoup(html, "html.parser")
    area = soup.select_one(RANK_AREA) or soup
    rows, seen = [], set()
    for a in area.select(SONG_LINK):
        img = a.select_one(COVER_IMG)
        alt = img.get("alt", "") if img else ""
        if ALT_SEP not in alt or a.get("href") in seen:
            continue
        seen.add(a.get("href"))
        title, artist = alt.split(ALT_SEP, 1)
        rows.append({"순위": len(rows) + 1,
                     "곡명": title.strip(),
                     "가수": artist.strip(),
                     "점수": text_of(a, SCORE_SEL),
                     "변동": change_of(a),
                     "링크": urljoin(ICHART_URL, a.get("href"))})
        if len(rows) == TOP_N:
            break
    return pd.DataFrame(rows)


df = parse_ranking(html)
df

,순위,곡명,가수,점수,변동,링크
0,1,NEON,"A2O, A2O LITE",211,-,https://www.ichart.kr/song/52712
1,2,꿈꾸던 어른이 되었나요?,머무르,132,-,https://www.ichart.kr/song/49735
2,3,퇴사할게여 (Narr. 기안84),소연 ((여자)아이들),103,-,https://www.ichart.kr/song/51466
3,4,LOVE ATTACK,RESCENE(리센느),69,-,https://www.ichart.kr/song/430
4,5,BAD,ATEEZ(에이티즈),60,▲1,https://www.ichart.kr/song/49081
5,6,Looks Can Kill,IVE(아이브),56,▼1,https://www.ichart.kr/song/52949
6,7,REDRED,CORTIS (코르티스),46,▼1,https://www.ichart.kr/song/46859
7,8,갑자기,아이오아이 (I.O.I),46,-,https://www.ichart.kr/song/47844
8,9,Deja Vu,RESCENE(리센느),43,▼1,https://www.ichart.kr/song/37116
9,10,Pretty Girl,RESCENE(리센느),42,▼1,https://www.ichart.kr/song/49465


> 0곡이 나오면? 사이트 구조가 바뀐 것입니다. F12로 다시 보고 위의 선택자를 고칩니다.

## ② 변환: 순위 표 → Claude → 카드뉴스 문구

순위 표를 글자로 바꿔 프롬프트에 넣고, JSON으로만 답하게 합니다.

In [7]:
MODEL = "claude-haiku-5-5"  # 모델 이름은 docs.claude.com 에서 확인

PROMPT = """아이차트 실시간 음악 순위다.
인스타그램 카드뉴스 문구를
JSON으로만 답할 것.
{{"제목": "...",
 "카드": ["1장 문구", "..."],
 "해시태그": ["#..."]}}
카드는 3~5장, 장마다 두 문장 이내.
순위표에 없는 사실은 쓰지 않기.

{table}"""

table = df[["순위", "곡명", "가수"]].to_string(index=False)
print(PROMPT.format(table=table))

아이차트 실시간 음악 순위다.
인스타그램 카드뉴스 문구를
JSON으로만 답할 것.
{"제목": "...",
 "카드": ["1장 문구", "..."],
 "해시태그": ["#..."]}
카드는 3~5장, 장마다 두 문장 이내.
순위표에 없는 사실은 쓰지 않기.

 순위                 곡명            가수
  1               NEON A2O, A2O LITE
  2      꿈꾸던 어른이 되었나요?           머무르
  3 퇴사할게여 (Narr. 기안84)  소연 ((여자)아이들)
  4        LOVE ATTACK  RESCENE(리센느)
  5                BAD   ATEEZ(에이티즈)
  6     Looks Can Kill      IVE(아이브)
  7             REDRED CORTIS (코르티스)
  8                갑자기 아이오아이 (I.O.I)
  9            Deja Vu  RESCENE(리센느)
 10        Pretty Girl  RESCENE(리센느)


In [8]:
import json

import anthropic

# ```json 울타리를 벗기고 json.loads, 실패하면 원문 보관
def parse_json(text):
    text = text.strip().removeprefix("```json").removeprefix("```")
    text = text.removesuffix("```").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("JSON 해석 실패 — 원문을 그대로 저장")
        return {"제목": "(JSON 해석 실패)", "카드": [text], "해시태그": []}


try:
    client = anthropic.Anthropic()  # ANTHROPIC_API_KEY 사용
    msg = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": PROMPT.format(table=table)}],
    )
    card = parse_json(msg.content[0].text)
except Exception as e:  # 키 없음·한도 초과·네트워크
    print("Claude 호출 실패:", e)
    card = None

card

Claude 호출 실패: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for the `Authorization` header to be explicitly omitted"


## ③ 저장: 엑셀 파일 (시트 두 개)

카드뉴스 표에는 출처와 "AI 생성 초안" 표시를 꼭 넣습니다.

In [9]:
SOURCE = "출처: 아이차트(ichart.kr)"


# 카드뉴스 딕셔너리 → 표
def card_table(card):
    if card is None:
        rows = [("안내", "문구 생성 실패 — 위 셀의 오류 확인")]
    else:
        rows = [("제목", card.get("제목", ""))]
        for i, text in enumerate(card.get("카드", []), start=1):
            rows.append((f"카드 {i}", text))
        rows.append(("해시태그", " ".join(card.get("해시태그", []))))
    rows.append(("출처", SOURCE))
    rows.append(("비고", "AI 생성 초안 · 검수 후 게시"))
    return pd.DataFrame(rows, columns=["구분", "문구"])


card_df = card_table(card)
card_df

,구분,문구
0,안내,문구 생성 실패 — 위 셀의 오류 확인
1,출처,출처: 아이차트(ichart.kr)
2,비고,AI 생성 초안 · 검수 후 게시


In [10]:
from datetime import date

out_dir = BASE_DIR / "output"
out_dir.mkdir(exist_ok=True)
path = out_dir / f"ichart_{date.today():%Y-%m-%d}.xlsx"

with pd.ExcelWriter(path) as writer:
    df.to_excel(writer, sheet_name="순위", index=False)
    card_df.to_excel(writer, sheet_name="카드뉴스", index=False)

print("저장 완료:", path)

저장 완료: c:\Users\artis\Downloads\crawlday_code\output\ichart_2026-10-28.xlsx


## ④ 다음 단계: 매일 자동 실행

노트북은 실습용입니다. 매일 자동으로 돌릴 때는 같은 내용을 한 파일로 묶은 `t01_ichart_demo.py`를
`run_daily.bat`과 Windows 작업 스케줄러로 실행합니다. 결과와 오류는 `logs/daily.log`에 남습니다.

In [ ]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

pw=await async_playwright().start()
browser=await pw.chromium.launch(headless=False, timeout=10000)
page=await browser.new_page()
await page.goto("https://www.ichart.kr/rank")
#await page.wait_for_selector('[aria-label="View All"]')
btn_view_all=page.get_by_text("View All").first
await btn_view_all.scroll_into_view_if_needed()
await btn_view_all.click()
await page.wait_for_timeout(1000)
html=await page.content()
html

#await browser.close()
#await pw.stop()
#